# 📚 RAG (Retrieval-Augmented Generation): Zero to Hero — A Guided Lab

RAG grounds an LLM's answers in **your own documents**: retrieve relevant chunks, then generate
an answer using them as context. It slashes hallucination and lets a model answer about private
or current data it was never trained on.

**Runs 100% offline** — a `MockLLM` + a from-scratch retriever. The architecture is identical to
production RAG (real embeddings + vector DB + real LLM).

**Prerequisite:** the Embeddings & Search lab (RAG = that retriever + a generation step).

**How this lab works** — 📖 Theory → 🧠 Mental model → 🖼️ ASCII diagram → 🔬 Example →
⚡ Pro tips → ⚠️ Traps → ✏️ Your Turn → ✅ Solution.

**Roadmap**
1. Why RAG? (the hallucination problem)
2. The RAG pipeline end-to-end
3. Ingest & chunk documents
4. Embed & index (the vector store)
5. Retrieve relevant chunks
6. Build a grounded prompt
7. Generate the answer
8. Guardrails: "I don't know" & citations
9. Evaluating a RAG system
10. 🏆 Capstone: a complete RAG assistant


In [ ]:
import numpy as np, re
from collections import Counter

class MockLLM:
    """Stand-in LLM. In production: OpenAI/Gemini. It answers ONLY from provided context."""
    def generate(self, prompt):
        # extract the context block and question to simulate grounded answering
        ctx = re.search(r"Context:(.*?)Question:", prompt, re.S)
        q = re.search(r"Question:(.*)", prompt, re.S)
        context = ctx.group(1).strip() if ctx else ""
        question = q.group(1).strip().lower() if q else ""
        if not context or "no relevant" in context.lower():
            return "I don't know based on the available information."
        # naive: echo the most relevant context sentence containing a query keyword
        sentences = [s.strip() for s in re.split(r"[.\n]", context) if s.strip()]
        for kw in tokenize(question):
            for s in sentences:
                if kw in s.lower():
                    return f"Based on the documentation: {s}."
        return sentences[0] + "." if sentences else "I don't know."

def tokenize(t): return re.findall(r"[a-z]+", t.lower())
llm = MockLLM()
print("Mock LLM ready.")

---
## Chapter 1 — Why RAG? The Hallucination Problem

📖 **Theory.** LLMs only "know" what was in their training data, and they can state falsehoods
**confidently** (hallucinate). They also can't know your private docs or anything after their
cutoff. **RAG** fixes this by *retrieving* relevant text and instructing the model to answer
**only from that text**.

🖼️ **Diagram — plain LLM vs RAG**
```
 Plain:  question ─────────────────► LLM ─► (may hallucinate)

 RAG:    question ─► retrieve docs ─► [context+question] ─► LLM ─► grounded answer
```

🧠 **Mental model.** RAG turns a "closed-book exam" (LLM guessing from memory) into an
"open-book exam" (LLM answering from provided pages).


In [ ]:
# Without RAG, a model has no access to YOUR policies:
print(llm.generate("Question: what is our refund window?"))  # no context -> "I don't know"

### ✏️ Your Turn 1.1
In one comment, list two situations where RAG is essential (think: private data, freshness).

In [ ]:
# 1. ...
# 2. ...


✅ **Solution**
```python
# 1. Answering over private/internal documents the model never trained on.
# 2. Answering about current/changing info (prices, policies) after the model's cutoff.
```

---
## Chapter 2 — The RAG Pipeline End-to-End

📖 **Theory.** Every RAG system is the same five stages:

🖼️ **Diagram — the RAG pipeline**
```
  ┌──────────┐   ┌───────┐   ┌────────┐   ┌──────────┐   ┌──────────┐
  │ 1 INGEST │─► │2 CHUNK│─► │3 EMBED │─► │4 RETRIEVE│─► │5 GENERATE│
  │ raw docs │   │ split │   │ +index │   │ top-k    │   │ grounded │
  └──────────┘   └───────┘   └────────┘   └──────────┘   └──────────┘
     (offline: build the index)          (online: per query)
```

🧠 **Mental model.** Stages 1–3 happen **once** (build the knowledge index). Stages 4–5 happen
**per question**. Keep that split clear in your head and in your code.


In [ ]:
# Our knowledge base: company policy documents
documents = [
    "Returns are accepted within 30 days of purchase with a valid receipt. Items must be unused.",
    "Refunds are processed within 5 to 7 business days to the original payment method.",
    "Subscriptions can be cancelled anytime from Account Settings under the Billing tab.",
    "The mobile app requires iOS 15 or later, or Android 10 or later, to install.",
    "For password resets, use the Forgot Password link on the login page.",
    "Standard shipping takes 3 to 5 business days within the continental United States.",
    "International orders may incur customs fees that are not included at checkout.",
    "Business accounts include a dedicated support line available 9am to 6pm Eastern.",
]
print(f"{len(documents)} policy documents")

### ✏️ Your Turn 2.1
Which two stages are "offline" (run once when docs change) and which are "online" (run per
query)? Answer in a comment.

In [ ]:
# offline stages: ...
# online stages: ...


✅ **Solution**
```python
# offline (build index once): ingest, chunk, embed+index
# online (per query): retrieve, generate
```

---
## Chapter 3 — Ingest & Chunk

📖 **Theory.** Long documents must be split into **chunks** so retrieval returns focused
passages that fit the LLM's context. Use **overlap** so a fact spanning a boundary isn't lost.
Our docs are short (already one chunk each), but we'll build a real chunker for longer text.

🖼️ **Diagram — why chunk?**
```
 huge 50-page doc ──► retrieve returns the WHOLE thing ──► context overflow ✗
 chunked passages ──► retrieve returns the 2 relevant paragraphs ──► focused ✓
```


In [ ]:
def chunk_document(text, chunk_size=25, overlap=5):
    words = text.split()
    chunks, start = [], 0
    while start < len(words):
        chunks.append(" ".join(words[start:start+chunk_size]))
        start += chunk_size - overlap
    return chunks

long_policy = ("Our warranty covers manufacturing defects for one year from the date of "
    "purchase. It does not cover accidental damage, water damage, or normal wear and tear. "
    "To make a warranty claim, contact support with your order number and a description of "
    "the issue. Approved claims are repaired or replaced free of charge within two weeks.")
for i, ch in enumerate(chunk_document(long_policy, 20, 5)):
    print(f"chunk {i}: {ch}\n")

⚠️ **Common trap.** Over-large chunks bury the relevant sentence among irrelevant text
(hurting retrieval precision); over-small chunks lose surrounding context. 200–500 words with
~10–20% overlap is a common starting point — tune with evaluation (Ch.9).

### ✏️ Your Turn 3.1
Chunk `long_policy` with `chunk_size=15, overlap=3`. How many chunks result?

In [ ]:
chunks = None
print(len(chunks) if chunks else None)

✅ **Solution**
```python
chunks = chunk_document(long_policy, 15, 3)
print(len(chunks))
```

---
## Chapter 4 — Embed & Index (the Vector Store)

📖 **Theory.** Each chunk becomes a **vector** (embedding). Store all vectors in an **index**
(a "vector store"). At query time we embed the query and find the nearest chunk vectors. We use a
TF-IDF-style embedding here; production uses a neural embedding model + a vector DB (Chroma,
FAISS, Pinecone).

🖼️ **Diagram — the vector store**
```
 chunk 0 ─► [0.2, 0.0, 0.9, ...]  ┐
 chunk 1 ─► [0.0, 0.7, 0.1, ...]  ├─ vector index (searchable by similarity)
 chunk 2 ─► [0.8, 0.0, 0.0, ...]  ┘
```


In [ ]:
# Build a simple embedding over the document vocabulary (stand-in for a real model)
vocab = sorted(set(w for d in documents for w in tokenize(d)))
vidx = {w:i for i,w in enumerate(vocab)}
N = len(documents)
df = Counter()
for d in documents:
    for w in set(tokenize(d)): df[w]+=1

def embed(text):
    v = np.zeros(len(vocab))
    for w, c in Counter(tokenize(text)).items():
        if w in vidx:
            idf = np.log((1+N)/(1+df[w])) + 1
            v[vidx[w]] = c*idf
    n = np.linalg.norm(v)
    return v/n if n>0 else v

class VectorStore:
    def __init__(self):
        self.chunks, self.vectors = [], []
    def add(self, chunk):
        self.chunks.append(chunk); self.vectors.append(embed(chunk))
    def build(self, chunks):
        for c in chunks: self.add(c)
        self.matrix = np.array(self.vectors)

store = VectorStore()
store.build(documents)   # each doc is one chunk here
print("indexed", len(store.chunks), "chunks, vector dim", store.matrix.shape[1])

### ✏️ Your Turn 4.1
Add the chunks of `long_policy` (from Ch.3) into a **new** `VectorStore` and report the total
number of indexed chunks.

In [ ]:
store2 = VectorStore()
# build with chunk_document(long_policy, 20, 5)
print(len(store2.chunks) if store2.chunks else None)

✅ **Solution**
```python
store2 = VectorStore()
store2.build(chunk_document(long_policy, 20, 5))
print(len(store2.chunks))
```

---
## Chapter 5 — Retrieve Relevant Chunks

📖 **Theory.** Embed the query, score every chunk by **cosine similarity**, return the top-k.
This is the "R" in RAG — if retrieval is bad, no amount of clever prompting saves the answer.

⚡ **Pro tip.** Retrieve a few *more* chunks than you think you need (k=4–8) and let the LLM
sift; but not so many that you bury the key chunk or blow the context budget.


In [ ]:
def cosine(a, b):
    d = np.linalg.norm(a)*np.linalg.norm(b)
    return 0.0 if d==0 else float(np.dot(a,b)/d)

def retrieve(query, store, k=3):
    q = embed(query)
    scored = [(chunk, cosine(q, vec)) for chunk, vec in zip(store.chunks, store.vectors)]
    return sorted(scored, key=lambda x: -x[1])[:k]

print("Query: 'how long do refunds take?'\n")
for chunk, score in retrieve("how long do refunds take", store, k=3):
    print(f"  {score:.3f}  {chunk[:70]}...")

### ✏️ Your Turn 5.1
Retrieve the top-2 chunks for `"what devices does the app support"` and confirm the iOS/Android
requirement doc is #1.

In [ ]:
# retrieve top-2 for the device-support query


✅ **Solution**
```python
for chunk, s in retrieve("what devices does the app support", store, k=2):
    print(round(s,3), chunk[:60])
```

---
## Chapter 6 — Build a Grounded Prompt

📖 **Theory.** The prompt must **clearly separate** retrieved context from the question, and
**instruct** the model to answer *only* from context and to decline if the answer isn't there.
This single instruction is your most important hallucination guardrail.

🖼️ **Diagram — grounded prompt structure**
```
 ┌─────────────────────────────────────────────┐
 │ Instruction: answer ONLY from context.       │
 │ If not in context, say "I don't know".       │
 │                                              │
 │ Context:                                     │
 │  - <chunk 1>                                 │
 │  - <chunk 2>                                 │
 │                                              │
 │ Question: <user question>                    │
 └─────────────────────────────────────────────┘
```


In [ ]:
def build_prompt(query, retrieved_chunks):
    context = "\n".join(f"- {c}" for c, _ in retrieved_chunks)
    return (
        "Answer the question using ONLY the context below. "
        'If the answer is not in the context, say "I don\'t know based on the available information."\n\n'
        f"Context:\n{context}\n\n"
        f"Question: {query}"
    )

chunks = retrieve("how long do refunds take", store, k=3)
prompt = build_prompt("how long do refunds take", chunks)
print(prompt)

⚠️ **Common trap.** If you don't explicitly forbid outside knowledge, the model may "help" by
adding facts not in your docs — reintroducing hallucination. Always pin it to the context.

### ✏️ Your Turn 6.1
Modify `build_prompt` to also ask the model to **cite** which context bullet it used (e.g.
"[from context item 2]").

In [ ]:
# write build_prompt_cited(query, retrieved_chunks)


✅ **Solution**
```python
def build_prompt_cited(query, retrieved_chunks):
    context = "\n".join(f"[{i}] {c}" for i,(c,_) in enumerate(retrieved_chunks))
    return ("Answer ONLY from context and cite the [number] you used. "
            'If not present, say "I don\'t know".\n\n'
            f"Context:\n{context}\n\nQuestion: {query}")
```

---
## Chapter 7 — Generate the Answer

📖 **Theory.** Feed the grounded prompt to the LLM. Now retrieval + generation combine into one
`answer()` function — the heart of RAG.


In [ ]:
def answer(query, store, k=3):
    retrieved = retrieve(query, store, k=k)
    prompt = build_prompt(query, retrieved)
    response = llm.generate(prompt)
    return response, retrieved

for q in ["how long do refunds take?",
          "what is the return window?",
          "how do I cancel my subscription?"]:
    resp, used = answer(q, store)
    print(f"Q: {q}\nA: {resp}\n")

⚡ **Pro tip.** Always **log the retrieved chunks** alongside every answer. When an answer is
wrong, 90% of the time the bug is in *retrieval* (wrong chunks), not generation — and you can
only see that if you logged them.

### ✏️ Your Turn 7.1
Ask `answer()` about shipping time, and print both the answer AND the retrieved chunks so you can
verify the answer is grounded in them.

In [ ]:
# call answer() for a shipping question; print response and used chunks


✅ **Solution**
```python
resp, used = answer("how long does shipping take?", store)
print("Answer:", resp)
for c, s in used: print("  used:", round(s,3), c[:50])
```

---
## Chapter 8 — Guardrails: "I don't know" & Similarity Threshold

📖 **Theory.** A good RAG system **declines** when it lacks the answer instead of inventing one.
The cheapest guardrail: if the top retrieved chunk's similarity is **below a threshold**, skip
generation and return "I don't know" — the docs simply don't cover it.

🖼️ **Diagram — the threshold gate**
```
 retrieve top-1 similarity
        │
   sim ≥ threshold ? ──yes─► generate grounded answer
        │
        └──no──► "I don't know" (don't risk a hallucination)
```


In [ ]:
def answer_guarded(query, store, k=3, threshold=0.05):
    retrieved = retrieve(query, store, k=k)
    top_score = retrieved[0][1] if retrieved else 0.0
    if top_score < threshold:
        return "I don't know based on the available information.", retrieved, top_score
    prompt = build_prompt(query, retrieved)
    return llm.generate(prompt), retrieved, top_score

# in-scope question -> answered
resp, _, score = answer_guarded("what is the return window?", store)
print(f"in-scope (top sim {score:.3f}): {resp}\n")

# out-of-scope question -> declined
resp, _, score = answer_guarded("do you offer gift wrapping?", store)
print(f"out-of-scope (top sim {score:.3f}): {resp}")

### ✏️ Your Turn 8.1
Find a question that is clearly **not** covered by the documents (e.g. about warranty length, if
you didn't index `long_policy`). Confirm `answer_guarded` returns "I don't know" with a low top
similarity.

In [ ]:
# try answer_guarded on an out-of-scope question and inspect the top score


✅ **Solution**
```python
resp, _, score = answer_guarded("what colors does the laptop come in?", store)
print(resp, "(top sim:", round(score,3), ")")
```

---
## Chapter 9 — Evaluating a RAG System

📖 **Theory.** Evaluate two things separately:
- **Retrieval quality** — did the right chunk get retrieved? (hit-rate / precision@k)
- **Answer quality** — is the answer correct & grounded? (exact/keyword match, or an LLM judge)

🖼️ **Diagram — two-part evaluation**
```
 query ─► RETRIEVE ─► [chunks]   ──► retrieval metric (was the right chunk here?)
                          │
                          ▼
                      GENERATE ─► answer ──► answer metric (correct? grounded?)
```


In [ ]:
# eval set: question -> (index of the correct doc, a keyword the answer should contain)
eval_set = [
    ("how long do refunds take?", 1, "5 to 7"),
    ("what is the return window?", 0, "30 days"),
    ("how do I cancel my subscription?", 2, "account settings"),
    ("how long does shipping take?", 5, "3 to 5"),
]

def evaluate_rag(store):
    retrieval_hits, answer_hits = 0, 0
    for q, correct_idx, keyword in eval_set:
        retrieved = retrieve(q, store, k=3)
        retrieved_idxs = [store.chunks.index(c) for c, _ in retrieved]
        if correct_idx in retrieved_idxs:
            retrieval_hits += 1
        resp, _ = answer(q, store)
        if keyword.lower() in resp.lower():
            answer_hits += 1
    n = len(eval_set)
    return retrieval_hits/n, answer_hits/n

ret_acc, ans_acc = evaluate_rag(store)
print(f"retrieval hit-rate@3: {ret_acc:.0%}")
print(f"answer keyword-match: {ans_acc:.0%}")

### ✏️ Your Turn 9.1
Add one more `(question, correct_idx, keyword)` case to `eval_set` (e.g. about the app's device
requirements) and re-run `evaluate_rag`. Did scores change?

In [ ]:
# append a new eval case and re-run evaluate_rag(store)


✅ **Solution**
```python
eval_set.append(("what OS does the app need?", 3, "ios 15"))
print(evaluate_rag(store))
```

---
## 🏆 Chapter 10 — Capstone: A Complete RAG Assistant

Wrap the whole pipeline into a `RAGAssistant` class: ingest docs → chunk → embed/index →
guarded retrieve+generate → return answer with the chunks it used. Build it before peeking.

In [ ]:
# Your RAGAssistant here
class RAGAssistant:
    def __init__(self, documents, chunk_size=25, overlap=5, threshold=0.05):
        pass
    def ask(self, query, k=3):
        pass

# bot = RAGAssistant(documents)
# print(bot.ask("how long do refunds take?"))


✅ **Capstone Solution**
```python
class RAGAssistant:
    def __init__(self, documents, chunk_size=25, overlap=5, threshold=0.05):
        # ingest + chunk
        self.chunks = []
        for doc in documents:
            self.chunks.extend(chunk_document(doc, chunk_size, overlap) if len(doc.split())>chunk_size else [doc])
        # embed + index
        self.store = VectorStore(); self.store.build(self.chunks)
        self.threshold = threshold
    def ask(self, query, k=3):
        retrieved = retrieve(query, self.store, k=k)
        top = retrieved[0][1] if retrieved else 0.0
        if top < self.threshold:
            return {"answer": "I don't know based on the available information.",
                    "chunks": [c for c,_ in retrieved], "confidence": round(top,3)}
        resp = llm.generate(build_prompt(query, retrieved))
        return {"answer": resp, "chunks": [c for c,_ in retrieved], "confidence": round(top,3)}

bot = RAGAssistant(documents)
import json
print(json.dumps(bot.ask("how long do refunds take?"), indent=2))
print(json.dumps(bot.ask("do you sell gift cards?"), indent=2))   # -> I don't know
```

🎉 **You built a complete RAG system!** Ingest → chunk → embed → retrieve → grounded generate,
with an "I don't know" guardrail and real evaluation. To go to production: swap `embed()` for a
real embedding model, `VectorStore` for Chroma/FAISS/Pinecone, and `MockLLM` for OpenAI/Gemini —
the pipeline shape stays exactly the same.

---
### 📌 Concept Quick-Reference
**Pipeline:** ingest → chunk → embed → index → retrieve → generate
**Chunking:** chunk_size + overlap; sentence-aware for prose
**Vector store:** embed chunks once, search by cosine similarity per query
**Grounded prompt:** separate Context from Question; forbid outside knowledge; allow "I don't know"
**Guardrails:** similarity threshold gate; citations; log retrieved chunks
**Evaluation:** retrieval hit-rate@k (right chunk?) + answer correctness/grounding
**To productionize:** real embeddings + vector DB (Chroma/FAISS/Pinecone) + real LLM
